In [ ]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Go get the data from https://drive.google.com/drive/folders/1xcdvCxNAEEfJElt7sEP-xT8dMKxn1_Lz

save to your own drive and update paths accordingly (needs to be unzipped, preferably the audio is all in one directory directly) - you can focus on the control and the lable groups, maybe ignore paired speech?

In [ ]:
import os
import librosa
import IPython.display as ipd
from IPython.display import display, clear_output
import pandas as pd
import numpy as np
from google.colab import drive
import ipywidgets as widgets
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas


SAMPLE_RATE = 44100
FFT_SIZE = 2048
FFT_HOP_SIZE = 512
LEN_SEC = 1.0 # when changing len_sec, need to update OUTPUT_DIR name
FRAME_LENGTH = int(44100 * LEN_SEC)
HOP_LENGTH = int(1.00 * FRAME_LENGTH) # no overlap
WINDOW = 'hann'

# Mount Google Drive
drive.mount('/content/drive')

# Set your audio folder inside Google Drive
RAW_AUDIO_DIR = '/content/drive/MyDrive/DATASETS/gtsinger/raw_audio/'

# Set the output CSV file
LABELS_FILE = "/content/drive/My Drive/DATASETS/emotions_dataset/gtsinger_emotions.csv"
OUT_DIR = '/content/drive/MyDrive/DATASETS/emotion_dataset'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Get a list of audio files
audio_files = []

for root, _, files in os.walk(RAW_AUDIO_DIR):
  for file in files:
    if file.endswith('.wav'):
      path = os.path.join(root, file)
      audio_files.append([file, path])

# Load existing labels if they exist
if os.path.exists(LABELS_FILE):
    labels_df = pd.read_csv(LABELS_FILE)
else:
    labels_df = pd.DataFrame(columns=["file", "subdir", "start_sec", "end_sec", "anger", "contempt", "disgust", "enjoyment", "fear", "sadness", "surprise"])

# Filter out already labeled files
labeled_files = set(labels_df["file"].unique()) if not labels_df.empty else set()
audio_files = [f for f in audio_files if os.path.basename(f[1]) not in labeled_files]

total_files = len(audio_files)

In [ ]:
# Global variables
current_file_index = 0
current_segment_start = 0
current_audio_data = None
current_file_name = ''
sr = None
segment_length = 0.5
segment_hop = 0.125
file_num = 1
segment_num = 1
segment_audio = None

# Label options
label_options = ["anger", "contempt", "disgust", "enjoyment", "fear", "sadness", "surprise", "delete"]
checkboxes = {label: widgets.Checkbox(value=False, description=label) for label in label_options}
label_box = widgets.VBox(list(checkboxes.values()))
confirm_button = widgets.Button(description="Confirm Labels")
output = widgets.Output()

def load_next_segment():
    """Loads the next segment of the audio file"""
    global current_segment_start, current_audio_data, current_file_name, sr, total_files, file_num, segment_num, segment_audio

    if current_segment_start >= librosa.get_duration(y=current_audio_data, sr=sr):
        save_and_next_file()
        return

    clear_output(wait=True)
    print(f"\nNow labeling: {current_file_name} ({librosa.get_duration(y=current_audio_data, sr=sr):.2f} sec)")
    print(f"{file_num}/{total_files} files")

    # Calculate start and end time
    start_time = current_segment_start
    end_time = min(current_segment_start + segment_length, librosa.get_duration(y=current_audio_data, sr=sr))

    # Extract segment
    start_sample = int(start_time * sr)
    end_sample = int(end_time * sr)
    segment_audio = current_audio_data[start_sample:end_sample]

    # Display audio
    print(f"\nSegment: {start_time:.2f} - {end_time:.2f} sec")
    ipd.display(ipd.Audio(segment_audio, rate=sr))

    # Display label checkboxes and button
    display(label_box)
    display(confirm_button)
    display(output)

    current_segment_start += segment_hop  # Move to next segment
    segment_num += 1

def save_labels(b, sr=SAMPLE_RATE, n_fft=FFT_SIZE, hop_length=FFT_HOP_SIZE,
                      window=WINDOW, frame_length=FRAME_LENGTH, frame_hop=HOP_LENGTH):
    """Saves the selected labels for the current segment and moves to the next"""
    global current_segment_start, labels_df, segment_audio

    # Get selected labels
    selected_labels = [label for label, cb in checkboxes.items() if cb.value]

    # Ensure labels are included
    if selected_labels:
      # Create a dictionary for the new row
      new_label = {
          "file": os.path.basename(audio_files[current_file_index][1]),
          "start_sec": current_segment_start - segment_length,
          "end_sec": current_segment_start,
      }

      folder_label = ""
      # Assign 1 to selected labels, 0 to unselected labels
      for style in styles:
        if style in selected_labels:
          new_label[style] = 1
          if folder_label != "":
            folder_label = folder_label + "_"
          folder_label = folder_label + style
        else:
           new_label[style] = 0

      # Convert to DataFrame
      new_label_df = pd.DataFrame([new_label])

      # Append to CSV (creates if it doesn't exist, adds header if needed)
      new_label_df.to_csv(LABELS_FILE, mode="a", header=not os.path.exists(LABELS_FILE), index=False)

      if 'delete' not in selected_labels:
        filename = os.path.splitext(os.path.basename(current_file_name))[0]
        out_file = f"{filename}_{segment_num}.wav"
        output_folder = os.path.join(OUT_DIR, folder_label)

        # Create folder if it doesn't exist
        if not os.path.exists(output_folder):
            os.makedirs(output_folder)

        output_path = os.path.join(output_folder, out_file)

        # Save audio segment
        sf.write(output_path, segment_audio, sr)

    # Clear checkboxes for next selection
    for cb in checkboxes.values():
        cb.value = False

    # Move to next segment
    load_next_segment()

def save_and_next_file():
    """Marks the current file as labeled and moves to the next"""
    global current_file_index, current_segment_start, file_num, segment_num

    print(f"\nFinished labeling: {audio_files[current_file_index][0]}")

    # Move to next file
    current_file_index += 1
    current_segment_start = 0
    segment_num = 1

    if current_file_index < len(audio_files):
        file_num+=1
        load_audio_file()
    else:
        print("\nAll files labeled!")

def load_audio_file():
    """Loads the next audio file for labeling"""
    global current_audio_data, sr, current_segment_start, current_file_name

    clear_output(wait=True)
    file_path = audio_files[current_file_index][1]

    # Load audio
    current_audio_data, sr = librosa.load(file_path, sr=None)
    current_file_name = os.path.split(file_path)[-1]

    # Load first segment
    load_next_segment()

# Bind button to function
confirm_button.on_click(save_labels)

In [ ]:
# Start labeling
if audio_files:
    load_audio_file()
else:
    print("No audio files found!")


Now labeling: f2_scales_breathy_u.wav (7.82 sec)
2/80 files

Segment: 0.00 - 0.50 sec


Button(description='Confirm Labels', style=ButtonStyle())

Output()